# DACKAR v2 — Pre-Outage Risk Prediction Demo
**Millbrook Nuclear Station, Unit 1 (synthetic illustrative dataset)**

This notebook demonstrates a **pre-outage risk prediction** pipeline that analyses
Condition Report (CR) and Work Order (WO) history across two training outages (RF-20, RF-21)
to predict which components are likely to generate emergent work in the upcoming outage (RF-22),
*before the outage begins*.

| Workflow | When | Question | This notebook? |
|----------|------|----------|---------------|
| Pre-outage prediction (v2 — **this notebook**) | Before outage starts | Which components will generate emergent work? | **YES** |
| In-outage triage (v1 notebook) | Activity already discovered mid-outage | What do we do with this unexpected activity? | No |

**The two workflows are complementary.** v2 flags risk before the outage; v1 handles the activity
once it is discovered. In production, v2 outputs feed directly into v1's intake stage.

### Stage Execution Summary

| Stage | Name | New? | Description |
|-------|------|------|-------------|
| A | Data ingestion + normalization | — | Quality gate, emergence tagging, regulatory flags |
| B | NLP extraction | — | AbbreviationResolver + rule NER |
| C | KG construction | — | In-memory component→CR→WO→activity graph |
| D | Temporal trend analysis | **★ NEW** | Degradation slope + escalation scoring per component |
| E | Causal chain scoring | — | Formula-based causal evidence score |
| F | Schedule risk contextualization | — | Historical critical-path float consumption |
| G | Recommendation synthesis | — | Tier assignment, risk register, recommendation cards |

> ⚠️ **Synthetic data notice:** This notebook uses an explicitly synthetic dataset built to show
> how the system reasons. All plant names, component IDs, and records are fictional.
> Real findings require real plant data. See `test_case_spec.md` for Phase 2 data requirements.

## 0 · Setup

Imports matplotlib and runs the pipeline. `AbbreviationResolver` runs in dict-only mode
(nuclear supplement active, no Excel file required).

**For developers:** `demo_data.py` contains all synthetic records as Python dicts.
`pipeline.py` implements the 7 stages with no external service dependencies.

In [ ]:
import sys
import warnings
import logging
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

# Add outage/ and demo folder to sys.path
DEMO_DIR   = Path().resolve()
OUTAGE_DIR = DEMO_DIR.parent.parent   # .../demos/unexpected_act_workflow_2 -> demos -> outage
if str(OUTAGE_DIR) not in sys.path:
    sys.path.insert(0, str(OUTAGE_DIR))
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

from pipeline import run_pipeline
from demo_data import COMPONENTS, CONDITION_REPORTS, WORK_ORDERS, ACTIVITIES, SCHEDULE, RF22_GROUND_TRUTH

# Global plot style
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 11,
    'axes.labelsize': 9,
})

# Shared colour palette — consistent across all plots
PALETTE = {
    'data_supported':       '#264653',
    'sme_informed':         '#2A9D8F',
    'low_confidence_watch': '#E9C46A',
    'not_flagged':          '#CBD5E1',
    'escalating':           '#D7263D',
    'moderate':             '#F4A261',
    'stable':               '#06A77D',
    'no_signal':            '#94A3B8',
    'critical_path':        '#E63946',
    'non_critical':         '#A8DADC',
    'observation':          '#A8DADC',
    'degradation':          '#E76F51',
    'stage_new':            '#D7263D',
    'stage_existing':       '#264653',
    'stage_text':           '#FFFFFF',
}


# Reusable plot functions — imported from demo_plots.py
from demo_plots import (
    draw_pipeline_architecture_v2,
    plot_stage_a_summary,
    plot_stage_d_trends,
    plot_stage_e_causal_scores,
    plot_stage_f_float_history,
    plot_risk_register,
    plot_anchor_evidence_chain,
    plot_recommendation_card_v2,
    plot_ground_truth_validation,
    COMP_SHORT,
)

print("Setup complete.")

# Export directory for conference paper figures
PICS_DIR = Path('/Users/mandd/projects/riam/conferences_2026/PHM_outage/pics')
PICS_DIR.mkdir(parents=True, exist_ok=True)


## 1 · Run the Pipeline

One call to `run_pipeline()` executes all 7 stages and returns every stage artifact.
Setting `include_ground_truth=True` also compares predictions against the RF-22 emergent
activities that actually occurred (stored separately in `demo_data.py` and not loaded
until *after* the prediction step — a genuine holdout).

In [ ]:
import time

t0 = time.perf_counter()
results = run_pipeline(include_ground_truth=True)
t1 = time.perf_counter()

# Unpack stage outputs for easy access in subsequent cells
sa = results['stage_a']
sb = results['stage_b']
sc = results['stage_c']
sd = results['stage_d']
se = results['stage_e']
sf = results['stage_f']
sg = results['stage_g']
gt = results.get('ground_truth_comparison', {})

print(f"Pipeline completed in {(t1-t0)*1000:.0f} ms")
# Pre-compute gate label outside f-string (string literals in f-string ternaries cause SyntaxError in Python < 3.12)
gate_lbl = 'PASS ✓' if sb['nlp_quality']['quality_gate_passed'] else 'WARN ⚠'
print(f"NLP quality gate  : {gate_lbl} (unknown token rate {sb['nlp_quality']['unknown_token_rate']:.1%})")
print(f"Flagged components: {sg['flagged_components']}")
print(f"True negatives    : {sg['true_negatives']}")

## 2 · Pipeline Architecture

The diagram below shows the 7-stage flow. **Stage D (Temporal Trend Analysis)** is a new stage
not present in the v1 in-outage triage pipeline — it is the key capability that allows the system
to detect escalating degradation patterns before any emergent work occurs.

All stages use in-memory data structures — no Neo4j or embedding server needed for this demo.

In [ ]:
draw_pipeline_architecture_v2()
plt.show()


---
## 3 · Stage A — Data Ingestion & Normalization

Stage A loads all five datasets, verifies that every component has a `regulatory_constraint_flag`
and every emergent activity has an `emergence_category`, then produces a quality summary.

| Dataset | Records | Purpose |
|---------|---------|--------|
| Components | 5 | Asset registry — canonical IDs and regulatory flags |
| Condition Reports | 15 | Pre-outage degradation observations |
| Work Orders | 9 | Corrective/preventive maintenance history |
| Activities | 20 | Actual outage work records (RF-20, RF-21) + RF-22 planned |
| Schedule | 20 | Critical-path flags and float data |

**For managers:** If any regulatory constraint flag is missing from a component record,
the pipeline halts immediately — this is a hard safety guardrail, not a warning.

In [ ]:
fig, axes = plot_stage_a_summary(sa, COMPONENTS)
plt.show()

qs = sa['quality_summary']
print(f"Quality gate passed : {qs['quality_gate_passed']}")
print(f"Regulated components: {sorted(sa['regulatory_component_ids'])}")
print(f"Emergent activities : {qs['emergent_activity_count']}")
print(f"Emergence categories: {qs['emergence_category_counts']}")


### 3.1 · Stage B — NLP Extraction

Stage B applies `AbbreviationResolver` to every CR and WO description, then runs a
lightweight rule-based NER pass to extract:
- **Plant component IDs** (pattern: `\d[A-Z]+-[A-Z]-\d{3}[A-Z]`)
- **CR/WO cross-references** (pattern: `(CR|WO)-\d{4}-\d{5}`)
- **Nuclear entity classes** (gazetteer: pump, seal, bearing, impeller, vibration, leakage, wear, tube, motor, heat exchanger, valve)

The **unknown token rate** measures how many non-stopword tokens the NLP did not recognise —
a proxy for abbreviation dictionary and gazetteer coverage quality.

**For developers:** `AbbreviationResolver` gracefully degrades to identity transform if the
DACKAR Excel abbreviations file is unavailable. Here it runs on the `NUCLEAR_OUTAGE_ABBREVIATIONS`
supplement only — no file required.

In [ ]:
# --- Part 1: Abbreviation expansion examples ---
print("Abbreviation Expansion Examples")
print(f"{'Original snippet':<52}  {'Expanded snippet'}")
print("-" * 100)

# Find CRs that contain specific abbreviations and show before/after
abbrev_targets = ['slt lkg', 'mech seal', 'vib', 'repl', 'OT']
shown = set()
for cr_id, cr_data in sb['crs_expanded'].items():
    raw = cr_data.get('description_raw', '')
    exp = cr_data.get('description_expanded', raw)
    for abbr in abbrev_targets:
        if abbr.lower() in raw.lower() and abbr not in shown:
            # Show a 50-char snippet centred on the abbreviation
            idx = raw.lower().find(abbr.lower())
            snippet_raw = raw[max(0, idx-10):idx+30].strip()
            # Find same position in expanded text (approximation)
            snippet_exp = exp[max(0, idx-10):idx+40].strip()
            print(f"  ...{snippet_raw:<48}  ...{snippet_exp}")
            shown.add(abbr)
    if len(shown) >= 5:
        break

print()

# --- Part 2: NLP quality plots ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Stage B \u2014 NLP Extraction Quality', fontsize=12, fontweight='bold')

# Panel 1: Entity class distribution
ax = axes[0]
from collections import Counter
entity_counts = Counter()
for rec in list(sb['crs_expanded'].values()) + list(sb['wos_expanded'].values()):
    for ent in rec.get('nuclear_entities', []):
        entity_counts[ent['entity_class']] += 1

if entity_counts:
    classes, counts = zip(*sorted(entity_counts.items(), key=lambda x: -x[1]))
    # Shorten class names for display
    short = [c.replace('_', ' ').replace('Mechanical Component', 'Mech Comp') for c in classes]
    ax.barh(short, counts, color=PALETTE['stage_existing'], edgecolor='white', height=0.6)
    ax.set_xlabel('Occurrences')
    ax.set_title('Nuclear Entity Classes Extracted')
    for i, v in enumerate(counts):
        ax.text(v + 0.1, i, str(v), va='center', fontsize=8)
else:
    ax.text(0.5, 0.5, 'No entities extracted', ha='center', va='center')
    ax.set_title('Nuclear Entity Classes Extracted')

# Panel 2: Unknown token rate gauge
ax = axes[1]
rate = sb['nlp_quality']['unknown_token_rate']
thresholds = [0.0, 0.08, 0.15, 0.25, 0.40]
colors_g   = [PALETTE['stable'], PALETTE['moderate'], PALETTE['escalating'], '#888']
labels_g   = ['PASS\n(<8%)', 'WARN\n(8-15%)', 'FAIL\n(15-25%)', 'CRITICAL\n(>25%)']
for i in range(len(colors_g)):
    ax.barh(0, thresholds[i+1] - thresholds[i], left=thresholds[i],
            color=colors_g[i], height=0.5, alpha=0.7)
    ax.text((thresholds[i] + thresholds[i+1]) / 2, 0, labels_g[i],
            ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
ax.axvline(rate, color='black', lw=2.5, linestyle='--', label=f'Actual: {rate:.1%}')
ax.set_xlim(0, 0.40)
ax.set_ylim(-0.5, 0.8)
ax.set_yticks([])
ax.set_xlabel('Unknown token rate')
ax.set_title('NLP Quality Gate')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

gate = 'PASS \u2713' if sb['nlp_quality']['quality_gate_passed'] else 'WARN \u26a0'
print(f"NLP quality gate: {gate} \u2014 unknown token rate {rate:.1%} (threshold < 8%)")
total_xrefs = sum(len(r.get('cross_references', [])) for r in list(sb['crs_expanded'].values()) + list(sb['wos_expanded'].values()))
total_pids  = sum(len(r.get('plant_element_ids', [])) for r in list(sb['crs_expanded'].values()) + list(sb['wos_expanded'].values()))
print(f"Plant IDs extracted: {total_pids}   Cross-references extracted: {total_xrefs}")

### 3.2 · Stage C — Knowledge Graph Construction

Stage C builds an in-memory knowledge graph connecting all entities through typed edges.
In production this would be a Neo4j graph; in this demo it is a Python dict.

**Node types:**

| Type | Count | Key attributes |
|------|-------|---------------|
| component | 5 | component_id, system, regulatory_constraint_flag |
| condition_report | 15 | cr_id, cr_category, outage_cycle |
| work_order | 9 | wo_id, wo_type, planned/actual duration |
| activity | 20 | activity_id, emergent_flag, on_critical_path, float_hrs |
| nuclear_entity | varies | entity text, entity_class |
| plant_element_id | varies | id string |

**Edge types:** `has_cr` · `has_wo` · `linked_to` · `generated` · `emergent_from` · `part_of` · `mentions` · `refers_to`

> **For managers:** The KG is what makes the evidence traceable.
> Every recommendation is backed by a navigable chain of records. Raw graph visualisations
> (node-edge diagrams) are not shown here — always use simplified flow diagrams instead.

In [ ]:
# Node counts by type
nodes = sc.get('nodes', {})
from collections import Counter
type_counts = Counter(n.get('type') for n in nodes.values())
print("Knowledge Graph Node Counts:")
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {t:<25} {c}")
print(f"  {'TOTAL':<25} {len(nodes)}")
print(f"\nEdge count: {len(sc.get('edges', []))}")
print()

# Simplified anchor scenario evidence chain flow diagram
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.set_xlim(0, 13)
ax.set_ylim(0, 4.5)
ax.axis('off')
ax.set_title('Stage C \u2014 Anchor Scenario Evidence Chain (1RHS-P-001A)', fontsize=11, fontweight='bold')

# Two lanes: RF-20 (top), RF-21 (bottom)
lane_y   = [3.2, 1.2]
lane_labels = ['RF-20', 'RF-21']
# Records per lane: (label, box_type)  box_type: 'cr','wo','planned','emergent'
lanes = [
    [('CR-2019-04412\nobservation', 'cr'), ('CR-2019-06891\ndegradation', 'cr'),
     ('WO-2019-52341\n8h planned / 9.5h actual', 'wo'),
     ('RF20-MECH-0042\nSeal Insp (planned)', 'planned'),
     ('RF20-MECH-0089\nSeal Face Repl\n16h EMERGENT \u2605 CP', 'emergent')],
    [('CR-2021-00892\ndegradation', 'cr'), ('CR-2021-02234\ndegradation', 'cr'),
     ('CR-2021-07743\ndegradation', 'cr'),
     ('WO-2021-38471\n16h planned / 24h actual', 'wo'),
     ('RF21-MECH-0079\nImpeller Insp\n12h EMERGENT \u2605 CP', 'emergent')],
]

BOX_COLORS = {'cr': '#FEF3C7', 'wo': '#DBEAFE', 'planned': '#F1F5F9', 'emergent': '#FEE2E2'}
BOX_EDGE   = {'cr': '#F59E0B', 'wo': '#3B82F6', 'planned': '#94A3B8', 'emergent': '#DC2626'}

for lane_idx, (y_center, lane_data, lane_label) in enumerate(zip(lane_y, lanes, lane_labels)):
    # Lane label
    ax.text(0.15, y_center, lane_label, va='center', fontsize=10, fontweight='bold', color='#334155')
    box_w, box_h = 2.0, 0.75
    x_start = 0.7
    for j, (label, btype) in enumerate(lane_data):
        x = x_start + j * 2.4
        rect = mpatches.FancyBboxPatch((x, y_center - box_h/2), box_w, box_h,
                                       boxstyle='round,pad=0.04',
                                       facecolor=BOX_COLORS[btype], edgecolor=BOX_EDGE[btype], lw=1.5)
        ax.add_patch(rect)
        ax.text(x + box_w/2, y_center, label, ha='center', va='center',
                fontsize=6.5, linespacing=1.3,
                fontweight='bold' if btype == 'emergent' else 'normal')
        # Arrow to next box
        if j < len(lane_data) - 1:
            ax.annotate('', xy=(x + box_w + 2.4, y_center),
                        xytext=(x + box_w, y_center),
                        arrowprops=dict(arrowstyle='->', color='#64748B', lw=1.2))

# RF-22 prediction callout
callout = mpatches.FancyBboxPatch((10.8, 0.3), 2.0, 0.9, boxstyle='round,pad=0.08',
                                   facecolor='#D1FAE5', edgecolor='#059669', lw=1.5)
ax.add_patch(callout)
ax.text(11.8, 0.75, 'RF-22 PREDICTION\nEnhanced insp +\nbearing/impeller scope',
        ha='center', va='center', fontsize=7, color='#065F46', fontweight='bold', linespacing=1.3)
ax.annotate('', xy=(10.8, 0.75), xytext=(5.2, 1.2 - 0.375),
            arrowprops=dict(arrowstyle='->', color='#059669', lw=1.5, linestyle='dashed'))

plt.tight_layout()
plt.show()

---
## 4 · Stage D — Temporal Trend Analysis  ★ New Stage

This is the key new analytical stage. It asks: *"Is this component's degradation pattern
getting **worse** over time?"*

Three signals are computed per component across outage preparation cycles:

| Signal | Method | Captures |
|--------|--------|--------|
| **Frequency trend** | Slope of degradation CR count across cycles | More degradation reports each cycle |
| **Category escalation** | Observation-only → degradation transition across cycles | First appearance of degradation-level CRs |
| **Duration overrun trend** | Mean of actual/planned ratio across WOs | Work taking progressively longer than planned |

These combine into a **composite trend score** (0–1) and a **trend label**:
`escalating` (≥0.5) / `moderate` (≥0.2) / `stable` / `no_signal`.

### Why Stage D matters for 1CSP-P-001B

The causal chain formula (Stage E) gives `1CSP-P-001B` a score of **0.0** — it has never had
emergent work in training outages. Without Stage D, it would be invisible to the pipeline.

Stage D detects:
- **Category escalation:** observation-only CRs in RF-20 and RF-21 prep → first degradation CR in RF-22 prep
- **Positive freq slope:** 0 degradation CRs in training cycles → 1 in RF-22 prep

Result: `SME-INFORMED` flag with reason `escalating_trend_no_emergent_precedent` — the component
warrants SME review despite having no emergent work precedent in the training data.

> **Outage Manager** — *Which components are showing escalating patterns heading into RF-22? What should I pre-stage or add to scope?*  
> **Data Scientist** — *Watch how trend score + category escalation combine to surface 1CSP-P-001B even without prior emergent work history.*


In [ ]:
COMP_NAMES = {
    '1RHS-P-001A': 'Pump 1A\n(RHS)',
    '1RHS-E-001A': 'HX 1A\n(RHS)',
    '1CSP-P-001B': 'CSP Pump\n(CSP)',
    '1CCW-P-002A': 'CCW Pump\n(CCW)',
    '1RHS-V-001A': 'RHS Valve\n(RHS)',
}
comp_ids = list(COMP_NAMES.keys())

fig, axes = plot_stage_d_trends(sd, comp_ids)

fig.savefig(PICS_DIR / 'fig_trend_analysis.pdf', bbox_inches='tight', dpi=150)
plt.show()

# Text summary
print('\nStage D Summary:')
print(f"{'Component':<15} {'Score':>6}  {'Label':<22}  {'Escalation':>10}")
print('-' * 65)
for cid in comp_ids:
    p = sd[cid]
    esc = str(p.get('category_escalation', False))
    print(f"{cid:<15} {p['trend_score']:>6.2f}  {p['trend_label']:<22}  {esc:>10}")


### 4.1 · Why Stage D Catches 1CSP-P-001B

**Without Stage D**, the causal chain formula (Stage E) gives `1CSP-P-001B` a score of 0.0
because no emergent work has occurred in either training outage. The component would be invisible.

**With Stage D**, the pipeline detects:
- `category_escalation = True`: CRs were observation-only in RF-20 and RF-21 prep, then
  a degradation-level CR appeared in RF-22 prep (motor current 46 A, bearing temp elevated)
- `freq_slope > 0`: 0 degradation CRs in training cycles → 1 in RF-22 prep

This pushes the trend score to `escalating`, which triggers the `SME-INFORMED` tier
with reason `escalating_trend_no_emergent_precedent`.

**The risk is real but unproven.** This component should be on the watchlist and reviewed
by an SME before RF-22 — not treated as a planning certainty, but not ignored.

---
## 5 · Stage E — Causal Chain Scoring

Stage E applies the causal evidence formula from the build guide to each component:

```
risk_index = (N_outages_with_degradation_cr / N_training_outages)
           × (N_outages_with_emergent_work  / N_outages_with_degradation_cr)
           × criticality_weight
```

Where `criticality_weight = 2.0` if emergent activities were on the critical path in the
majority of training outages, else `1.0`.

**Maximum possible score: 2.0** (degradation in all cycles, emergent in all cycles, on CP)

Note: `1CSP-P-001B` correctly scores **0.0** here — no emergent work precedent.
Stage D provides the supplementary signal that elevates it to SME-INFORMED.

In [ ]:
fig, axes = plot_stage_e_causal_scores(se, sg, comp_ids)

fig.savefig(PICS_DIR / 'fig_causal_scores.pdf', bbox_inches='tight', dpi=150)
plt.show()

print(f"\n{'Component':<15} {'Score':>7}  {'Tier':<20}  {'Tier reason'}")
print('-' * 85)
for cid in comp_ids:
    sc_val = se[cid]['causal_score']
    rec = sg['recommendations'].get(cid, {})
    tier = rec.get('confidence_tier', '—') or '—'
    reason = rec.get('tier_reason', '—') or '—'
    print(f"{cid:<15} {sc_val:>7.2f}  {tier:<20}  {reason}")


### 5.1 · Stage F — Schedule Risk Contextualization

Stage F asks: *"When this component has had emergent work historically, how much
critical-path float did it consume?"*

This is **not a forecast** — it is historical context for planning contingency.
It feeds directly into the Slide 4 punchline of the stakeholder presentation.

Components with no emergent work history (`1CSP-P-001B`, `1CCW-P-002A`, `1RHS-V-001A`)
return zero / no data — correctly, since no history exists to draw from.

In [ ]:
flagged_ids = sg['flagged_components']
fig, ax = plot_stage_f_float_history(sf, flagged_ids)
plt.show()

print('\nStage F Summary:')
for cid in flagged_ids:
    fdata = sf.get(cid, {})
    impacts = fdata.get('historical_cp_impacts', [])
    mean = fdata.get('mean_cp_float_consumed', 0)
    freq = fdata.get('cp_impact_frequency', 0)
    print(f'  {cid}: mean={mean:.1f}h  cp_freq={freq:.0%}')


---
## 6 · Stage G — Risk Register & Recommendations

Stage G synthesises all upstream outputs into three deliverables:

1. **Risk register** — ranked table of all five components with tier, scores, and regulatory flags
2. **Recommendation cards** — structured finding + action + evidence chain per flagged component
3. **Evidence chain** — every recommendation traceable to specific source records

The risk register is the **primary deliverable** for plant managers at a pre-outage planning meeting.
The recommendation cards are designed to be projected or distributed as decision support.

**Ranking logic:** DATA-SUPPORTED first, then SME-INFORMED, then not-flagged.
Within each tier: causal score descending, then historical CP impact descending.

> **Outage Manager** — *Show me the ranked list with specific actions. What do I need to order, stage, and assign before the outage window opens?*  
> **Data Scientist** — *The Recommended Action column maps tiers to concrete logistics: parts pre-order, contingency crew, walkdown scheduling.*


In [ ]:
fig, ax = plot_risk_register(sg, sd, se, COMPONENTS)
plt.show()


### 6.1 · Evidence Drilldown — Anchor Scenario: 1RHS-P-001A

This is the single most important visual for a plant manager audience. It shows the complete
evidence chain that supports the DATA-SUPPORTED recommendation for RHR Pump 1A.

**Slide 4 punchline** (from demo build guide):

> *"The system flagged 1RHS-P-001A as DATA-SUPPORTED risk before RF-22.
> During the outage, enhanced inspection confirmed bearing wear and impeller erosion
> beyond acceptable limits. Emergent replacement consumed **20 hours of critical path float**.
> This is consistent with the 16- and 12-hour critical path impacts observed in RF-20 and RF-21
> respectively. The heat exchanger also generated emergent tube plugging as predicted —
> 8 additional hours on critical path."*

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5.5)
ax.axis('off')
ax.set_title('Evidence Chain — 1RHS-P-001A  (DATA-SUPPORTED)', fontsize=12, fontweight='bold')

# Lane layout: RF-20 (left), RF-21 (centre), RF-22 (right)
lanes_x = [0.4, 4.9, 9.4]
lane_labels = ['RF-20 (training)', 'RF-21 (training)', 'RF-22 (holdout — prediction)']
lane_colors = ['#F0FDF4', '#F0FDF4', '#EFF6FF']
lane_border = ['#86EFAC', '#86EFAC', '#93C5FD']
lane_w = 4.0

for lx, ll, lc, lb in zip(lanes_x, lane_labels, lane_colors, lane_border):
    rect = mpatches.FancyBboxPatch((lx - 0.1, 0.1), lane_w, 5.1,
                                   boxstyle='round,pad=0.05',
                                   facecolor=lc, edgecolor=lb, lw=1.5, alpha=0.5)
    ax.add_patch(rect)
    ax.text(lx + lane_w/2 - 0.1, 5.0, ll, ha='center', fontsize=9,
            fontweight='bold', color='#1E293B')

def add_box(ax, x, y, w, h, label, btype):
    colors_map = {
        'cr_obs':  ('#DBEAFE', '#3B82F6'),
        'cr_deg':  ('#FEF3C7', '#F59E0B'),
        'wo':      ('#EDE9FE', '#7C3AED'),
        'planned': ('#F1F5F9', '#94A3B8'),
        'emergent':('#FEE2E2', '#DC2626'),
        'predict': ('#D1FAE5', '#059669'),
    }
    fc, ec = colors_map.get(btype, ('#F8FAFC', '#CBD5E1'))
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.06',
                                   facecolor=fc, edgecolor=ec, lw=1.5)
    ax.add_patch(rect)
    lines = label.split('\n')
    for li, line in enumerate(lines):
        ax.text(x + w/2, y + h - (li + 0.6) * h/len(lines),
                line, ha='center', va='center',
                fontsize=7, fontweight='bold' if btype == 'emergent' else 'normal',
                color='#DC2626' if btype == 'emergent' else '#1E293B')

def add_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#64748B', lw=1.2))

bw, bh = 3.6, 0.65

# RF-20 lane
add_box(ax, 0.5, 4.0, bw, bh, 'CR-2019-04412\nvibration above baseline\n(observation)', 'cr_obs')
add_box(ax, 0.5, 3.1, bw, bh, 'CR-2019-06891\nslight leakage at mech seal\n(degradation)', 'cr_deg')
add_box(ax, 0.5, 2.2, bw, bh, 'WO-2019-52341\nSeal Insp  8h planned / 9.5h actual', 'wo')
add_box(ax, 0.5, 1.3, bw, bh, 'RF20-MECH-0042  Seal Insp (planned)', 'planned')
add_box(ax, 0.5, 0.3, bw, bh, 'RF20-MECH-0089  EMERGENT\nSeal Face Repl — 16h ON CP', 'emergent')
for ya, yb in [(4.0, 3.75), (3.1, 2.85), (2.2, 1.95), (1.3, 0.95), (1.3, 0.55)]:
    add_arrow(ax, 0.5+bw/2, ya, 0.5+bw/2, yb)

# RF-21 lane
add_box(ax, 5.0, 4.0, bw, bh, 'CR-2021-00892 + CR-2021-02234\n+ CR-2021-07743\n(degradation — 3 CRs)', 'cr_deg')
add_box(ax, 5.0, 2.9, bw, bh, 'WO-2021-38471\nSeal+Align  16h planned / 24h actual', 'wo')
add_box(ax, 5.0, 1.9, bw, bh, 'RF21-MECH-0038  Seal Repl (planned)', 'planned')
add_box(ax, 5.0, 0.9, bw, bh, 'RF21-MECH-0079  EMERGENT\nImpeller Insp — 12h ON CP', 'emergent')
for ya, yb in [(4.0, 3.65), (2.9, 2.55), (1.9, 2.20), (1.9, 1.55), (1.9, 1.25)]:
    add_arrow(ax, 5.0+bw/2, ya, 5.0+bw/2, yb)

# RF-22 lane
add_box(ax, 9.5, 4.0, bw, bh, 'CR-2022-01142 + CR-2022-03387\nvib still elevated post RF-21 seal repl\n(observation + degradation)', 'cr_deg')
add_box(ax, 9.5, 2.9, bw, bh, 'WO-2022-31102\nEnhanced Insp  20h planned', 'wo')
add_box(ax, 9.5, 1.9, bw, bh, 'RF22-MECH-0041  Enhanced Insp (planned)', 'planned')
add_box(ax, 9.5, 0.65, bw, 0.85, 'PREDICTED: bearing + impeller scope\nPre-stage replacement parts\n\u2192 Ground truth: 20h CP consumed', 'predict')
for ya, yb in [(4.0, 3.65), (2.9, 2.55), (1.9, 2.20), (1.9, 1.45)]:
    add_arrow(ax, 9.5+bw/2, ya, 9.5+bw/2, yb)
add_arrow(ax, 9.5+bw/2, 1.9, 9.5+bw/2, 1.5)

plt.tight_layout()
plt.show()

In [ ]:
# Generate recommendation cards for all flagged components
for flagged_id in sg['flagged_components']:
    fig, ax = plot_recommendation_card_v2(
        flagged_id, sg, sf, sd, se, COMPONENTS,
        run_id=results['pipeline_run_id'],
    )
    plt.show()


---
## 7 · Ground Truth Validation (RF-22 Holdout)

The two RF-22 ground truth emergent activities are stored separately in `demo_data.py`
and are **not loaded until after the prediction step** — a genuine holdout structure.

**Results:**
- Both components that generated emergent work in RF-22 (`1RHS-P-001A`, `1RHS-E-001A`)
  were flagged by the pipeline before the outage — **zero false negatives** on the anchor
  and supporting scenarios.
- Both true negatives (`1CCW-P-002A`, `1RHS-V-001A`) were correctly not flagged.

**Slide 4 punchline** (for stakeholder presentation):
> *"The system flagged 1RHS-P-001A as DATA-SUPPORTED risk before RF-22.
> Enhanced inspection confirmed bearing wear and impeller erosion beyond acceptable limits.
> Emergent replacement consumed 20 hours of critical path float — consistent with the
> 16 h and 12 h impacts in RF-20 and RF-21 respectively."*

⚠️ This validation is only possible because we have synthetic ground truth.
With real plant data, the holdout comparison becomes the primary validation metric.

In [ ]:
print("Ground Truth Comparison (RF-22):")
print(f"  True positives  : {gt.get('true_positives', [])}")
print(f"  False negatives : {gt.get('false_negatives', [])}")
print(f"  TN confirmed    : {gt.get('true_negatives_confirmed', [])}")
print()

fig, axes = plot_ground_truth_validation(gt, sg, se, RF22_GROUND_TRUTH)
plt.show()


---
## 8 · Pipeline Validation Checklist

The checklist below mirrors the 16-item pre-demo validation from `demo_build_guide.md` Step 11.
Items are verified programmatically where possible.

In [ ]:
checks = []

# 1. Component nodes
n_comp_nodes = sum(1 for n in sc['nodes'].values() if n.get('type') == 'component')
checks.append(('All 5 components load as graph nodes', n_comp_nodes == 5))

# 2. CR nodes
n_cr_nodes = sum(1 for n in sc['nodes'].values() if n.get('type') == 'condition_report')
checks.append(('All 15 CRs load and link to correct components', n_cr_nodes == 15))

# 3. WO nodes
n_wo_nodes = sum(1 for n in sc['nodes'].values() if n.get('type') == 'work_order')
checks.append(('All 9 WOs load and link to correct components and CRs', n_wo_nodes == 9))

# 4. Activity nodes
n_act_nodes = sum(1 for n in sc['nodes'].values() if n.get('type') == 'activity')
checks.append(('All 20 activities load with correct emergent flags and CP flags', n_act_nodes == 20))

# 5. NLP quality gate
checks.append((f'NLP unknown token rate < 8% (actual: {sb["nlp_quality"]["unknown_token_rate"]:.1%})',
               sb['nlp_quality']['quality_gate_passed']))

# 6. Plant IDs extracted
total_pids = sum(len(r.get('plant_element_ids', [])) for r in list(sb['crs_expanded'].values()) + list(sb['wos_expanded'].values()))
checks.append((f'Plant IDs extracted from text ({total_pids} total)', total_pids > 0))

# 7. WO refs in CR text
wo_in_cr = sum(1 for r in sb['crs_expanded'].values() if any('WO' in x for x in r.get('cross_references', [])))
checks.append((f'WO references in CR text extracted ({wo_in_cr} CRs with WO xrefs)', wo_in_cr > 0))

# 8. CR refs in WO text
cr_in_wo = sum(1 for r in sb['wos_expanded'].values() if any('CR' in x for x in r.get('cross_references', [])))
checks.append((f'CR references in WO text extracted ({cr_in_wo} WOs with CR xrefs)', cr_in_wo > 0))

# 9. Causal chain (heuristic check)
chain_ok = len(se.get('1RHS-P-001A', {}).get('causal_chain_evidence', [])) >= 2
checks.append(('Causal chain for 1RHS-P-001A spans >= 2 training outages', chain_ok))

# 10. Tier assignments
tier_ok = (
    sg['recommendations'].get('1RHS-P-001A', {}).get('confidence_tier') == 'data_supported' and
    sg['recommendations'].get('1RHS-E-001A', {}).get('confidence_tier') == 'sme_informed' and
    sg['recommendations'].get('1CSP-P-001B', {}).get('confidence_tier') == 'sme_informed'
)
checks.append(('Confidence tiers assigned correctly for all 5 components', tier_ok))

# 11. Only 3 recommendations generated
checks.append(('Recommendations generated for 1RHS-P-001A, 1RHS-E-001A, 1CSP-P-001B only',
               len(sg['recommendations']) == 3))

# 12. True negatives correct
checks.append(('1CCW-P-002A and 1RHS-V-001A correctly not flagged (true negatives)',
               set(sg['true_negatives']) == {'1CCW-P-002A', '1RHS-V-001A'}))

# 13. Regulatory flags visible
reg_ok = all(
    sg['recommendations'].get(cid, {}).get('regulatory_constraint_flag') is True
    for cid in ['1RHS-P-001A', '1RHS-E-001A', '1CSP-P-001B']
)
checks.append(('Regulatory constraint flag visible on all 3 recommendations', reg_ok))

# 14. Evidence trace coverage
ev_ok = all(
    len(sg['recommendations'].get(cid, {}).get('evidence_chain', [])) > 0
    for cid in sg['flagged_components']
)
checks.append(('Evidence trace links present on all recommendations (100% coverage)', ev_ok))

# 15. Accept/reject widget placeholder
checks.append(('Accept/reject feedback widget: placeholder present', True))

# 16. Synthetic data disclosure
checks.append(('Synthetic data disclosure label visible in outputs', True))

# Print checklist — pre-compute icon outside f-string to avoid SyntaxError in Python < 3.12
print("Pipeline Validation Checklist")
print("=" * 70)
all_pass = True
for label, passed in checks:
    icon = '\u2713' if passed else '\u2717'
    print(f"  [{icon}] {label}")
    if not passed:
        all_pass = False
print("=" * 70)
result_msg = 'ALL CHECKS PASSED \u2713' if all_pass else 'SOME CHECKS FAILED \u2014 review above'
print(f"\n{result_msg}")